In [1]:
#!pip install autogluon
#!pip install autots

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import seaborn as sns
import random
import os

In [ ]:
from pathlib import Path

dataset_name = "webbrowsing_train.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"])
print("Loaded:", data_file)

In [4]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
df = df.rename(columns={
    'ue_ident': 'item_id',
    'DATE': 'timestamp',
    'mac_dl_brate': 'target'
})

uni_data = TimeSeriesDataFrame(df)

In [6]:
prediction_length = 96

train_size = int(len(df) * 0.8)
train_data = uni_data.iloc[:train_size]
test_data = uni_data.iloc[train_size:]

In [ ]:
predictor = TimeSeriesPredictor(prediction_length=prediction_length,
                                         target="target",
                                         known_covariates_names=["mac_dl_cqi", "mac_dl_mcs", "mac_dl_ok",
                                                                 "mac_dl_nok"],
                                         eval_metric="MAE", freq='ms').fit(
    train_data,
    hyperparameters={
        "Chronos": [
            {"model_path": "bolt_small", "fine_tune": True, "ag_args": {"name_suffix": "FineTuned"}},
        ]
    },
    enable_ensemble=False,
    random_seed= 42,
)

In [ ]:
freq = test_data.index.levels[1].inferred_freq

future_rows = []

for item_id, df_item in test_data.groupby(level="item_id"):
    last_ts = df_item.index.get_level_values("timestamp").max()

    future_ts = pd.date_range(
        start=last_ts + pd.Timedelta(milliseconds=1),
        periods=predictor.prediction_length,
        freq="ms"
    )

    for ts in future_ts:
        future_rows.append((item_id, ts))

future_cov = pd.DataFrame(index=pd.MultiIndex.from_tuples(
    future_rows, names=["item_id", "timestamp"]
))


for col in predictor.known_covariates_names:
    future_cov[col] = None


for col in predictor.known_covariates_names:
    mean_val = test_data[col].mean()
    future_cov[col] = mean_val

In [11]:
def rolling_chronos_forecast_all(
    predictor,
    test_data,
    prediction_length=96,
    stride=1,
    measure_time=False
):

    results = []
    times = []

    if not isinstance(test_data.index, pd.MultiIndex):
        raise ValueError("Expected test_data with MultiIndex (item_id, timestamp).")

    # Extract frequency from the data
    freq = test_data.index.get_level_values("timestamp").to_series().diff().mode()[0]
    print(freq)

    for item_id, series in test_data.groupby(level="item_id"):
        series = series.reset_index()

        for start in range(0, len(series) - prediction_length, stride):

            # Rolling context
            context = series.iloc[: start + prediction_length]
            context_df = context.set_index(["item_id", "timestamp"])

            # Build future covariates for this window
            last_ts = context["timestamp"].iloc[-1]
            future_ts = pd.date_range(start=last_ts + pd.Timedelta(freq),
                                      periods=prediction_length,
                                      freq=freq)

            # Build covariate frame manually
            future_cov_df = pd.DataFrame({
                "item_id": item_id,
                "timestamp": future_ts,
            })

            # Fill all covariates using last known value
            for col in predictor.known_covariates_names:
                last_val = context[col].iloc[-1]
                future_cov_df[col] = last_val

            # Convert to TimeSeriesDataFrame
            future_cov_tdf = TimeSeriesDataFrame(
                future_cov_df.set_index(["item_id", "timestamp"])
            )

            # Predict
            t0 = time.time()
            forecast = predictor.predict(context_df, known_covariates=future_cov_tdf)
            t1 = time.time()

            if measure_time:
                times.append(t1 - t0)

            forecast_mean = forecast.loc[item_id]["mean"].to_numpy().flatten()

            df_forecast = pd.DataFrame({
                "item_id": item_id,
                "timestamp": future_ts,
                "mean": forecast_mean
            })

            results.append(df_forecast)

    forecasts_df = pd.concat(results, ignore_index=True)
    return (forecasts_df, times) if measure_time else forecasts_df

In [ ]:
rolling_preds= rolling_chronos_forecast_all(
    predictor, test_data, prediction_length=96, stride=1,measure_time=False
)


In [ ]:
actual = test_data.reset_index()[["item_id", "timestamp", "target"]]

aligned = rolling_preds.merge(
    actual, on=["item_id", "timestamp"], how="inner"
)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_data.reset_index()[["target"]])

aligned["mean_scaled"] = scaler.transform(aligned[["mean"]].to_numpy())
aligned["target_scaled"] = scaler.transform(aligned[["target"]].to_numpy())

rmse_scaled = np.sqrt(mean_squared_error(aligned["target_scaled"], aligned["mean_scaled"]))
mae_scaled = mean_absolute_error(aligned["target_scaled"], aligned["mean_scaled"])

print(f"Scaled RMSE: {rmse_scaled:.4f}")
print(f"Scaled MAE: {mae_scaled:.4f}")


In [ ]:
plt.plot(aligned["target"], label='Actual')
plt.plot(aligned["mean"], label='Predicted')
plt.legend()
plt.show()

In [17]:
def block_average(df, value_col, item_col="item_id", timestamp_col="timestamp", block_len=96):
    df = df.sort_values([item_col, timestamp_col]).reset_index(drop=True)
    results = []

    for item_id, group in df.groupby(item_col):
        y = group[value_col].to_numpy()
        ts = group[timestamp_col].to_numpy()

        num_blocks = len(y) // block_len
        if num_blocks == 0:
            continue

        y_trunc = y[:num_blocks*block_len].reshape(num_blocks, block_len)
        ts_trunc = ts[:num_blocks*block_len].reshape(num_blocks, block_len)

        results.append(pd.DataFrame({
            item_col: item_id,
            "block_id": np.arange(num_blocks),
            "value_avg": y_trunc.mean(axis=1),
            "timestamp_mid": ts_trunc[:, block_len // 2]  # middle timestamp
        }))

    return pd.concat(results, ignore_index=True)


In [18]:
pred_blocked = block_average(aligned, value_col="mean", block_len=prediction_length)
actual_blocked = block_average(aligned, value_col="target", block_len=prediction_length)


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_blocked['timestamp_mid'], actual_blocked['value_avg'], label='Actual')
plt.plot(pred_blocked['timestamp_mid'], pred_blocked['value_avg'], label='Predicted')
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.show()


In [ ]:
results_dir = Path("../results/metrics/web_browsing")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    "model": "Chronos-finetuning",
    "setting": "multivariate",
    "dataset": "web_browsing",
    "rmse": rmse_scaled,
    "mae": mae_scaled,
}])

metrics_file = results_dir / "chronosft_multi_metrics.csv"
metrics_df.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)